# Huấn luyện Mở rộng TemporalNet (17 COCO Keypoints) & Xuất ONNX cho Android Edge AI

Notebook này chạy trực tiếp trên **Kaggle GPU (T4 / P100)** mà **không cần tải bộ dữ liệu video nặng về máy tính cá nhân**.

### Quy trình tự động từ A-Z trong Notebook:
1. **Cài đặt thư viện**: `ultralytics`, `onnx`, `onnxruntime-gpu`.
2. **Quét & Lọc dữ liệu thông minh (`/kaggle/input`)**:
   - Tự động tìm tất cả video trong bộ **`payutch/fall-video-dataset`** (và **`GMDCSA24`** nếu có gắn kèm).
   - **Chỉ lấy nhánh `raw video`** (bỏ qua hoàn toàn các file CSV MediaPipe 33 điểm và các video đã vẽ đè khung xương).
   - **Chống rò rỉ dữ liệu (Group-Disjoint Split)**: Gom các góc máy cùng kịch bản (ví dụ 8 góc camera `chute01..24` hoặc cùng `Subject`) vào chung 1 nhóm để không bao giờ lọt cùng 1 cú ngã vừa ở tập Train vừa ở tập Test.
3. **Trích xuất 17 khớp xương COCO ở nhịp `10 Hz` bằng `yolo26n-pose.pt`** (hỗ trợ lưu cache để chạy tiếp nếu tạm dừng).
4. **Huấn luyện `TemporalNet` với Data Augmentation Lật Gương (Horizontal Flip)**:
   - Tự động nhân đôi dữ liệu Train bằng phép lật ngang khung hình và hoán đổi cặp khớp Trái - Phải chuẩn giải phẫu (`5↔6`, `7↔8`, `9↔10`, `11↔12`, `13↔14`, `15↔16`).
   - Giữ nguyên chuẩn đầu vào `(STEPS=30, FEATURES=93, HZ=10)` tương thích 100% với `realtime_yolo_fall_detection.py` trên PC và App Android.
5. **Đánh giá trên tập Test & Xuất 3 file kết quả trong `/kaggle/working/export_models/`**:
   - `temporal_fall.pt` (Thay thế vào `data/gmdcsa24_experiment/temporal_fall.pt` trên PC).
   - `temporal_fall.onnx` (Nhúng vào `assets/` của App Android Red Magic 7).
   - `yolo26n-pose.onnx` (Nhúng vào `assets/` của App Android Red Magic 7).

In [ ]:
# 1. Cài đặt thư viện cần thiết trên Kaggle
!pip install -q ultralytics onnx onnxruntime

In [ ]:
# 2. Kiểm tra GPU và chuẩn bị mô hình YOLO26n-Pose
import hashlib
import json
import math
import os
import re
import shutil
import time
from pathlib import Path

import cv2
import numpy as np
import torch
from torch import nn
from ultralytics import YOLO

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU Name: {torch.cuda.get_device_name(0)}')

WORK_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('./kaggle_working')
OUT_DIR = WORK_DIR / 'fall_experiment'
EXPORT_DIR = WORK_DIR / 'export_models'
OUT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Tải hoặc sử dụng yolo26n-pose.pt chuẩn của Ultralytics
YOLO_WEIGHTS = WORK_DIR / 'yolo26n-pose.pt'
if not YOLO_WEIGHTS.exists():
    # Kiểm tra xem người dùng có gắn kèm yolo26n-pose.pt trong /kaggle/input không
    found_weights = list(Path('/kaggle/input').rglob('yolo26n-pose.pt')) if Path('/kaggle/input').exists() else []
    if found_weights:
        shutil.copy2(found_weights[0], YOLO_WEIGHTS)
    else:
        _ = YOLO('yolo26n-pose.pt')
        if Path('yolo26n-pose.pt').exists():
            shutil.copy2('yolo26n-pose.pt', YOLO_WEIGHTS)

POSE_SHA256 = hashlib.sha256(YOLO_WEIGHTS.read_bytes()).hexdigest()
print(f'YOLO26n-Pose weights: {YOLO_WEIGHTS} (SHA256: {POSE_SHA256})')

In [ ]:
# 3. Quét tự động các video trong /kaggle/input (Lọc bỏ CSV MediaPipe & video vẽ đè xương)
VIDEO_EXTS = {'.mp4', '.avi', '.mov', '.mkv', '.webm'}
IGNORE_KEYWORDS = {'keypoint', 'skeleton', 'mediapipe', 'annotated', 'overlay', 'output', 'vis'}

def infer_label_and_group(video_path: Path):
    parts_lower = [p.lower() for p in video_path.parts]
    full_str = '/'.join(parts_lower)
    stem_lower = video_path.stem.lower()

    # Bỏ qua các thư mục hoặc file chứa kết quả vẽ xương / keypoint
    if any(kw in full_str for kw in IGNORE_KEYWORDS):
        return None, None

    # Xác định nhãn: 1 = Fall (Té ngã), 0 = Normal / ADL (Bình thường)
    non_fall_tokens = {'no_fall', 'nofall', 'non_fall', 'nonfall', 'not_fall', 'adl', 'normal', 'daily'}
    label = None
    for p in reversed(parts_lower):
        if any(nf in p for nf in non_fall_tokens):
            label = 0
            break
        if 'fall' in p:
            label = 1
            break
    if label is None:
        return None, None

    # Gom nhóm kịch bản (Group ID) để chống rò rỉ dữ liệu giữa 8 góc camera hoặc cùng Subject
    chute_match = re.search(r'(chute\s*\d+)', full_str)
    subj_match = re.search(r'(subject\s*\d+|subj[_\-]?\d+|person[_\-]?\d+|actor[_\-]?\d+)', full_str)
    if chute_match:
        group_id = 'mcfd_' + re.sub(r'\s+', '', chute_match.group(1))
    elif subj_match:
        group_id = 'subj_' + re.sub(r'\s+', '', subj_match.group(1))
    else:
        # Bỏ hậu tố góc camera như _cam1, _c02 để các góc máy của cùng 1 cảnh vào chung nhóm
        base_scene = re.sub(r'([_\-]?(cam|camera|view|angle)[_\-]?\d+)', '', stem_lower)
        group_id = f'{video_path.parent.name.lower()}_{base_scene[:18]}'

    return label, group_id

def discover_videos(input_root=Path('/kaggle/input'), max_videos=None):
    candidates = []
    if not input_root.exists():
        print('Không tìm thấy /kaggle/input, hãy gắn dataset payutch/fall-video-dataset ở cột bên phải!')
        return []

    all_files = sorted([p for p in input_root.rglob('*') if p.suffix.lower() in VIDEO_EXTS])
    print(f'Tìm thấy tổng cộng {len(all_files)} file video trong {input_root}...')

    seen_paths = set()
    for idx, vf in enumerate(all_files):
        resolved = str(vf.resolve())
        if resolved in seen_paths:
            continue
        label, group_id = infer_label_and_group(vf)
        if label is None:
            continue
        cap = cv2.VideoCapture(resolved)
        ok, frame = cap.read()
        fps = cap.get(cv2.CAP_PROP_FPS)
        frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
        cap.release()
        if not ok or fps <= 0 or frames < 5:
            continue
        seen_paths.add(resolved)
        short_hash = hashlib.md5(resolved.encode('utf-8')).hexdigest()[:8]
        rec_id = f"{group_id}_{'fall' if label == 1 else 'adl'}_{vf.stem}_{short_hash}"
        rec_id = re.sub(r'[^a-zA-Z0-9_\-]', '_', rec_id)
        candidates.append(dict(
            id=rec_id,
            subject=group_id,
            activity='Fall' if label == 1 else 'ADL',
            label=int(label),
            path=resolved,
            fps=float(fps),
            frames=int(frames),
            duration=float(frames / fps),
            width=int(frame.shape[1]),
            height=int(frame.shape[0])
        ))
        if max_videos and len(candidates) >= max_videos:
            break

    fall_cnt = sum(r['label'] == 1 for r in candidates)
    adl_cnt = sum(r['label'] == 0 for r in candidates)
    print(f'Đã lọc hợp lệ: {len(candidates)} video (Fall: {fall_cnt}, ADL/Normal: {adl_cnt})')
    return candidates

# Đặt MAX_VIDEOS = None để quét toàn bộ dataset, hoặc đặt số lượng (vd: 800) nếu muốn chạy nhanh hơn
MAX_VIDEOS = None
records = discover_videos(Path('/kaggle/input'), max_videos=MAX_VIDEOS)

# Chia tập Train (70%) / Validation (15%) / Test (15%) theo Group (không rò rỉ góc camera)
unique_groups = sorted({r['subject'] for r in records})
rng = np.random.RandomState(42)
rng.shuffle(unique_groups)
n_groups = len(unique_groups)
n_train = max(1, int(0.70 * n_groups))
n_val = max(1, int(0.15 * n_groups))

splits = {
    'train': unique_groups[:n_train],
    'validation': unique_groups[n_train:n_train + n_val],
    'test': unique_groups[n_train + n_val:] if n_train + n_val < n_groups else unique_groups[-1:]
}

manifest_path = OUT_DIR / 'manifest.json'
splits_path = OUT_DIR / 'splits.json'
manifest_path.write_text(json.dumps({'sample_hz': 10, 'window_seconds': 3.0, 'records': records}, indent=2))
splits_path.write_text(json.dumps(splits, indent=2))
for k, grp in splits.items():
    sub_recs = [r for r in records if r['subject'] in set(grp)]
    print(f"Split [{k}]: {len(sub_recs)} videos (Fall={sum(r['label']==1 for r in sub_recs)}, ADL={sum(r['label']==0 for r in sub_recs)})")

In [ ]:
# 4. Trích xuất chuỗi 17 điểm mốc COCO ở nhịp 10 Hz bằng YOLO26n-Pose
HZ = 10
STEPS = 30
FEATURES = 93

def iou(a, b):
    lo = np.maximum(a[:2], b[:2])
    hi = np.minimum(a[2:], b[2:])
    inter = np.prod(np.maximum(hi - lo, 0))
    area_a = np.prod(np.maximum(a[2:] - a[:2], 0))
    area_b = np.prod(np.maximum(b[2:] - b[:2], 0))
    return float(inter / max(area_a + area_b - inter, 1e-8))

def sample_video(path):
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    if not cap.isOpened() or fps <= 0:
        cap.release()
        return
    n = 0
    next_sample = 0.0
    try:
        while cap.grab():
            t = n / fps
            n += 1
            if t + 1e-8 < next_sample:
                continue
            ok, frame = cap.retrieve()
            if not ok:
                continue
            yield t, frame
            next_sample += 1.0 / HZ
    finally:
        cap.release()

def extract_video(model, path, device, batch_size=16):
    poses, boxes, timestamps = [], [], []
    batch, times = [], []
    previous = None
    frame_w, frame_h = 640, 480

    def process():
        nonlocal previous, frame_w, frame_h
        results = model.predict(batch, imgsz=640, conf=0.25, classes=[0], device=device, verbose=False)
        for t, frame, result in zip(times, batch, results):
            frame_h, frame_w = frame.shape[:2]
            scale = max(frame_h, frame_w)
            if result.keypoints is None or len(result.boxes) == 0:
                poses.append(np.zeros((17, 3), np.float32))
                boxes.append(np.zeros(4, np.float32))
                previous = None
            else:
                bb = result.boxes.xyxy.cpu().numpy()
                areas = np.prod(bb[:, 2:] - bb[:, :2], axis=1)
                overlaps = np.array([iou(previous, b) for b in bb]) if previous is not None else np.zeros(len(bb))
                index = int(overlaps.argmax()) if overlaps.max() >= 0.1 else int(areas.argmax())
                previous = bb[index].copy()
                kp = result.keypoints.data[index].cpu().numpy().copy()
                kp[:, :2] /= scale
                poses.append(kp.astype(np.float32))
                boxes.append((previous / scale).astype(np.float32))
            timestamps.append(t)
        batch.clear()
        times.clear()

    for t, frame in sample_video(path):
        times.append(t)
        batch.append(frame)
        if len(batch) >= batch_size:
            process()
    if batch:
        process()
    if not timestamps:
        return None
    return dict(
        pose=np.stack(poses),
        box=np.stack(boxes),
        time=np.array(timestamps, np.float64),
        width=np.array(frame_w, np.int32),
        height=np.array(frame_h, np.int32)
    )

kp_dir = OUT_DIR / 'keypoints'
kp_dir.mkdir(parents=True, exist_ok=True)
yolo_model = YOLO(str(YOLO_WEIGHTS))

valid_records = []
t0 = time.time()
for i, rec in enumerate(records, 1):
    dest = kp_dir / f"{rec['id']}.npz"
    sig = json.dumps({'model_sha256': POSE_SHA256, 'hz': HZ, 'imgsz': 640}, sort_keys=True)
    if dest.exists():
        valid_records.append(rec)
        continue
    data = extract_video(yolo_model, rec['path'], DEVICE, batch_size=16)
    if data is None or len(data['time']) < 5:
        continue
    np.savez_compressed(dest, **data, signature=np.array(sig))
    valid_records.append(rec)
    if i % 25 == 0 or i == len(records):
        elapsed = time.time() - t0
        print(f'[{i}/{len(records)}] Extracted {len(valid_records)} valid videos ({elapsed:.1f}s elapsed)')

records = valid_records
manifest_path.write_text(json.dumps({'sample_hz': HZ, 'window_seconds': STEPS / HZ, 'records': records}, indent=2))
print(f'Hoàn tất trích xuất Keypoints cho {len(records)} video!')

In [ ]:
# 5. Tiền xử lý đặc trưng 93 chiều, Lật gương (Horizontal Flip Augmentation) & Huấn luyện TemporalNet
FLIP_PAIRS = [(1, 2), (3, 4), (5, 6), (7, 8), (9, 10), (11, 12), (13, 14), (15, 16)]

def horizontal_flip_sample(data):
    """Tạo bản sao lật gương ngang (Mirror Augmentation) và hoán đổi cặp khớp Trái - Phải COCO."""
    pose = data['pose'].astype(np.float32).copy()
    box = data['box'].astype(np.float32).copy()
    w = float(data['width']) if 'width' in data else 640.0
    h = float(data['height']) if 'height' in data else 480.0
    w_norm = w / max(w, h)

    has_kp = pose[:, :, 2] > 0
    pose[:, :, 0] = np.where(has_kp, w_norm - pose[:, :, 0], 0.0)
    for left_idx, right_idx in FLIP_PAIRS:
        pose[:, [left_idx, right_idx]] = pose[:, [right_idx, left_idx]]

    has_box = box[:, 2] > box[:, 0]
    x1_new = np.where(has_box, np.maximum(0.0, w_norm - box[:, 2]), 0.0)
    x2_new = np.where(has_box, np.maximum(0.0, w_norm - box[:, 0]), 0.0)
    box[:, 0] = x1_new
    box[:, 2] = x2_new

    return dict(pose=pose, box=box, time=data['time'])

def features(data):
    """Hàm trích xuất 93 đặc trưng chuẩn đồng bộ 100% với gmdcsa_pipeline.py và App Android."""
    pose = data['pose'].astype(np.float32).copy()
    box = data['box'].astype(np.float32)
    good = pose[:, :, 2] >= 0.2
    valid = (box[:, 2] > box[:, 0]) & (good.sum(axis=1) >= 4)
    center = (box[:, :2] + box[:, 2:]) / 2
    hip = pose[:, [11, 12], :2].mean(axis=1)
    reliable_hip = good[:, 11] & good[:, 12]
    origin = np.where(reliable_hip[:, None], hip, center)
    size = np.maximum(np.linalg.norm(box[:, 2:] - box[:, :2], axis=1), 0.05)
    local = (pose[:, :, :2] - origin[:, None]) / size[:, None, None]
    local[~good] = 0
    confidence = pose[:, :, 2:3] * good[:, :, None]
    velocity = np.zeros_like(local)
    dt = np.maximum(np.diff(data['time']).astype(np.float32), 1e-3)
    velocity[1:] = (local[1:] - local[:-1]) / dt[:, None, None]
    velocity[1:] *= (good[1:] & good[:-1])[:, :, None]
    global_pos = np.concatenate([center, box[:, 2:] - box[:, :2]], axis=1)
    global_velocity = np.zeros_like(global_pos)
    global_velocity[1:] = (global_pos[1:] - global_pos[:-1]) / dt[:, None]
    global_velocity[1:] *= (valid[1:] & valid[:-1])[:, None]
    x = np.concatenate([
        local.reshape(-1, 34),
        confidence.reshape(-1, 17),
        velocity.reshape(-1, 34),
        global_pos,
        global_velocity
    ], axis=1)
    x[~valid] = 0
    return np.nan_to_num(x), valid.astype(np.float32)

def windows(data, stride=HZ):
    x, mask = features(data)
    starts = list(range(0, max(1, len(x) - STEPS + 1), stride))
    last = max(0, len(x) - STEPS)
    if starts[-1] != last:
        starts.append(last)
    bags, masks, ends = [], [], []
    for s in starts:
        xx, mm = x[s:s + STEPS], mask[s:s + STEPS]
        if len(xx) < STEPS:
            pad = STEPS - len(xx)
            xx = np.pad(xx, ((pad, 0), (0, 0)))
            mm = np.pad(mm, (pad, 0))
        bags.append(xx)
        masks.append(mm)
        ends.append(float(data['time'][min(s + STEPS - 1, len(x) - 1)]))
    return np.stack(bags), np.stack(masks), ends

class TemporalNet(nn.Module):
    def __init__(self):
        super().__init__()
        layers = []
        incoming = FEATURES
        for dilation in [1, 2, 4]:
            layers += [
                nn.Conv1d(incoming, 64, 3, padding=dilation, dilation=dilation),
                nn.ReLU(),
                nn.Dropout(0.2)
            ]
            incoming = 64
        self.temporal = nn.Sequential(*layers)
        self.head = nn.Linear(128, 1)

    def forward(self, x, mask):
        z = self.temporal(x.transpose(1, 2))
        m = mask[:, None, :]
        avg = (z * m).sum(2) / m.sum(2).clamp_min(1)
        peak = z.masked_fill(m == 0, -1e4).max(2).values
        peak = torch.where(m.sum(2) > 0, peak, torch.zeros_like(peak))
        return self.head(torch.cat([avg, peak], 1)).squeeze(1)

def metrics(labels, probabilities, threshold):
    y = np.array(labels, dtype=bool)
    pred = np.array(probabilities) >= threshold
    tp = int((pred & y).sum())
    tn = int((~pred & ~y).sum())
    fp = int((pred & ~y).sum())
    fn = int((~pred & y).sum())
    recall = tp / max(tp + fn, 1)
    specificity = tn / max(tn + fp, 1)
    precision = tp / max(tp + fp, 1)
    acc = (tp + tn) / max(tp + tn + fp + fn, 1)
    return dict(
        tp=tp, tn=tn, fp=fp, fn=fn,
        accuracy=acc,
        recall=recall,
        specificity=specificity,
        precision=precision,
        f1=2 * precision * recall / max(precision + recall, 1e-8),
        balanced_accuracy=(recall + specificity) / 2
    )

def score(model, bags):
    model.eval()
    results = []
    with torch.no_grad():
        for r, x, m, ends in bags:
            logits = model(x, m)
            usable = m.mean(1) >= 0.5
            probability = float(torch.sigmoid(logits[usable].max()).item()) if usable.any() else 0.0
            results.append(dict(
                id=r['id'], subject=r['subject'], label=r['label'],
                probability=probability, usable_windows=int(usable.sum().item())
            ))
    return results

# Nạp dữ liệu vào bộ nhớ và áp dụng Horizontal Flip cho tập Train
groups = {'train': [], 'validation': [], 'test': []}
split_lookup = {k: set(v) for k, v in splits.items()}

for rec in records:
    npz_path = kp_dir / f"{rec['id']}.npz"
    if not npz_path.exists():
        continue
    with np.load(npz_path, allow_pickle=False) as raw_npz:
        data = {k: raw_npz[k] for k in raw_npz.files}
    # Dùng stride=5 (0.5s) cho tập Train để bắt trọn khoảnh khắc ngã nhanh
    is_train = rec['subject'] in split_lookup['train']
    x, m, ends = windows(data, stride=5 if is_train else HZ)
    target_split = 'train' if is_train else ('validation' if rec['subject'] in split_lookup['validation'] else 'test')
    groups[target_split].append((rec, torch.tensor(x, device=DEVICE), torch.tensor(m, device=DEVICE), ends))

    if is_train:
        flipped_data = horizontal_flip_sample(data)
        xf, mf, ends_f = windows(flipped_data, stride=5)
        rec_flip = dict(rec, id=rec['id'] + '_flip')
        groups['train'].append((rec_flip, torch.tensor(xf, device=DEVICE), torch.tensor(mf, device=DEVICE), ends_f))

print(f"Số lượng mẫu (bags) sau Augmentation -> Train: {len(groups['train'])}, Val: {len(groups['validation'])}, Test: {len(groups['test'])}")

# Huấn luyện mạng TemporalNet
torch.manual_seed(42)
np.random.seed(42)
model = TemporalNet().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
best_val_loss = float('inf')
patience = 0
EPOCHS = 60
checkpoint_path = EXPORT_DIR / 'temporal_fall.pt'

for epoch in range(1, EPOCHS + 1):
    model.train()
    losses = []
    for i in np.random.permutation(len(groups['train'])):
        r, x, m, _ = groups['train'][i]
        usable = m.mean(1) >= 0.5
        if not usable.any():
            continue
        logits = model(x, m)[usable]
        bag_logit = logits.max()
        loss = nn.functional.binary_cross_entropy_with_logits(
            bag_logit, torch.tensor(float(r['label']), device=DEVICE)
        )
        if r['label'] == 0:
            loss = (loss + nn.functional.binary_cross_entropy_with_logits(logits, torch.zeros_like(logits))) / 2
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        losses.append(float(loss.item()))

    val_scores = score(model, groups['validation'])
    yy = np.array([r['label'] for r in val_scores])
    pp = np.clip([r['probability'] for r in val_scores], 1e-6, 1 - 1e-6)
    val_loss = float(-(yy * np.log(pp) + (1 - yy) * np.log(1 - pp)).mean())
    print(f'Epoch {epoch:02d}/{EPOCHS} | Train Loss: {np.mean(losses):.4f} | Val Loss: {val_loss:.4f}')

    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        patience = 0
        torch.save(dict(
            state_dict=model.state_dict(),
            hz=HZ,
            steps=STEPS,
            features=FEATURES,
            epoch=epoch,
            splits=splits,
            pose_model_sha256=POSE_SHA256,
            method='Multi-camera weak video labels + Horizontal Flip Augmentation, MIL max pooling'
        ), checkpoint_path)
    else:
        patience += 1
    if patience >= 12:
        print(f'Early stopping tại epoch {epoch}')
        break

# Nạp lại epoch tốt nhất và tìm ngưỡng threshold tối ưu trên tập Validation
saved = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
model.load_state_dict(saved['state_dict'])
val_scores = score(model, groups['validation'])
thresholds = np.linspace(0.15, 0.85, 71)
best_threshold = float(max(
    thresholds,
    key=lambda t: metrics([r['label'] for r in val_scores], [r['probability'] for r in val_scores], t)['balanced_accuracy']
))
saved['threshold'] = best_threshold
torch.save(saved, checkpoint_path)

print('\n================ KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH ================')
print(f'Best Epoch: {saved["epoch"]} | Optimal Threshold: {best_threshold:.2f}')
for split_name in ['validation', 'test']:
    sc = score(model, groups[split_name])
    m_res = metrics([r['label'] for r in sc], [r['probability'] for r in sc], best_threshold)
    print(f'[{split_name.upper()}] Accuracy: {m_res["accuracy"]*100:.2f}% | Recall: {m_res["recall"]*100:.2f}% | Precision: {m_res["precision"]*100:.2f}% | F1: {m_res["f1"]*100:.2f}%')

In [ ]:
# 6. Xuất 2 mô hình sang định dạng Mobile (.onnx) và kiểm tra sai số (Parity Verification)
import onnx
import onnxruntime as ort

class TemporalNetONNXWrapper(nn.Module):
    """Wrapper xuất trực tiếp xác suất té ngã (0.0 -> 1.0) cho App Android."""
    def __init__(self, base_model: TemporalNet):
        super().__init__()
        self.base = base_model

    def forward(self, x, mask):
        logits = self.base(x, mask)
        return torch.sigmoid(logits)

model_cpu = TemporalNet().cpu().eval()
model_cpu.load_state_dict(saved['state_dict'])
onnx_wrapper = TemporalNetONNXWrapper(model_cpu).eval()

dummy_x = torch.randn(1, STEPS, FEATURES, dtype=torch.float32)
dummy_mask = torch.ones(1, STEPS, dtype=torch.float32)
temporal_onnx_path = EXPORT_DIR / 'temporal_fall.onnx'

torch.onnx.export(
    onnx_wrapper,
    (dummy_x, dummy_mask),
    str(temporal_onnx_path),
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=['features', 'mask'],
    output_names=['fall_probability'],
    dynamic_axes={
        'features': {0: 'batch_size'},
        'mask': {0: 'batch_size'},
        'fall_probability': {0: 'batch_size'}
    }
)

# Kiểm tra khớp số học giữa PyTorch và ONNX Runtime
ort_sess = ort.InferenceSession(str(temporal_onnx_path), providers=['CPUExecutionProvider'])
with torch.no_grad():
    pt_out = onnx_wrapper(dummy_x, dummy_mask).numpy()
ort_out = ort_sess.run(None, {'features': dummy_x.numpy(), 'mask': dummy_mask.numpy()})[0]
max_diff = float(np.max(np.abs(pt_out - ort_out)))
print(f'Đã xuất {temporal_onnx_path.name} ({temporal_onnx_path.stat().st_size / 1024:.1f} KB) | Sai số PyTorch vs ONNX: {max_diff:.2e}')

# Xuất YOLO26n-Pose sang ONNX chuẩn cho Mobile
yolo_export = YOLO(str(YOLO_WEIGHTS))
exported_yolo = Path(yolo_export.export(format='onnx', imgsz=640, opset=17, simplify=True))
target_yolo_onnx = EXPORT_DIR / 'yolo26n-pose.onnx'
if exported_yolo.exists() and exported_yolo.resolve() != target_yolo_onnx.resolve():
    shutil.copy2(exported_yolo, target_yolo_onnx)

# Lưu kèm file cấu hình metadata cho App Android đọc ngưỡng threshold
mobile_meta = {
    'hz': HZ,
    'steps': STEPS,
    'features': FEATURES,
    'threshold': round(best_threshold, 4),
    'pose_model_sha256': POSE_SHA256,
    'input_size': 640
}
(EXPORT_DIR / 'model_metadata.json').write_text(json.dumps(mobile_meta, indent=2))

print('\n================ DANH SÁCH FILE TẢI VỀ TỪ KAGGLE ================')
for f in sorted(EXPORT_DIR.iterdir()):
    print(f' - {f.name} ({f.stat().st_size / 1024:.1f} KB)')